# Danh sách mô tả rỗng và nhãn trung tính

Phân tích thăm dò trên Qwen: bỏ ô giữ chỗ, thay tên mục giải phẫu bằng nhãn trung tính khớp token, và kết hợp hai thay đổi. Các chuỗi được nạp nguyên trạng từ tài nguyên đã kiểm tra; không dùng câu hỏi hoặc đáp án để dựng nhãn. Kết quả được báo cáo tách khỏi các kiểm định xác nhận.

In [ ]:
import os
import sys
from pathlib import Path

source_root = Path(os.environ.get("MEDVQA_SOURCE_ROOT", ".")).resolve()
if not (source_root / "medvqa").is_dir():
    raise FileNotFoundError("Set MEDVQA_SOURCE_ROOT to the extracted source directory.")
if str(source_root) not in sys.path:
    sys.path.insert(0, str(source_root))

## Cấu hình thí nghiệm

Chọn thư mục dữ liệu và nơi lưu kết quả của các điều kiện đối chứng. Dùng `PILOT` để kiểm tra kỹ thuật trước khi chạy `TEST`.

In [ ]:
from pathlib import Path

MODE = "PILOT"
RUN_ID = "pilot_neutral_controls"
ANSWERER = "Q"
CELLS = ["ITEMS", "NEUTRAL-DASH", "NEUTRAL-LIST"]
PLATFORM = None
DESIGN = "MAIN"
INPUT_ROOT = None
LOCK_FILE = None
RUN_PARAMS_FILE = None
OBS_DIRS = []
EDIT_DIRS = []
CORR_PLAN_FILE = None
LABEL_MAP_FILE = None
PRIOR_OUTPUT_DIRS = []
OUT_ROOT = None
SHARD_TIME_LIMIT_S = None
SHARD_INDEX, NUM_SHARDS = (0, 1)
MAX_ITEMS_THIS_SESSION = None
PRECISION = "PREC-0"
BACKEND = "hf"
LLAVAMED_CODE_COMMIT = None
LLAVAMED_CONV_MODE = None
INSTALL = False
MODULE_DIR = source_root

## Thực thi và lưu kết quả

In [ ]:
import sys
import time
from collections import Counter

sys.path.insert(0, str(MODULE_DIR))

In [ ]:
from medvqa import common as U
from medvqa import execution as RUN

PLATFORM = PLATFORM or RUN.MODEL_PLATFORM[ANSWERER]
PLATFORM_CFG = RUN.platform_setup(PLATFORM)
INPUT_ROOT = Path(INPUT_ROOT or PLATFORM_CFG["input_root"])
OUT_ROOT = Path(OUT_ROOT or PLATFORM_CFG["out_root"])
SHARD_TIME_LIMIT_S = SHARD_TIME_LIMIT_S or PLATFORM_CFG["time_limit_s"]
LOCK_FILE = Path(LOCK_FILE or INPUT_ROOT / "." / "locked_inputs.sha256")
RUN_PARAMS_FILE = Path(RUN_PARAMS_FILE or INPUT_ROOT / "." / "run_params.json")
if DESIGN == "MAIN" and ANSWERER == "M":
    raise ValueError("MAIN: ANS-M (LLaVA-Med) is not run")
ENV = RUN.env_for(ANSWERER, PLATFORM)
if ANSWERER == "M" and BACKEND == "hf" and (not LLAVAMED_CONV_MODE):
    raise ValueError(
        "Set LLAVAMED_CONV_MODE from the official eval script at the pinned commit"
    )
RUN.verify_run_params_first(LOCK_FILE, RUN_PARAMS_FILE, MODE)
platform_report = RUN.check_platform(ANSWERER, PLATFORM, MODE, RUN_PARAMS_FILE)
RUN.check_design(DESIGN, MODE, RUN_PARAMS_FILE)
ENV_PINS = RUN.env_pins_for(MODE, RUN_PARAMS_FILE, ENV)
if INSTALL and BACKEND == "hf":
    RUN.install_env(ENV, ENV_PINS, LLAVAMED_CODE_COMMIT)
env_report = RUN.verify_env(
    ENV, ENV_PINS, MODE, LLAVAMED_CODE_COMMIT, check_git=BACKEND == "hf"
)
print("environment:", env_report)

In [ ]:
from medvqa import conditions as CL
from medvqa import interventions as I
from medvqa import models as B
from medvqa import prompts as P

PREP = INPUT_ROOT / "prep_out"
PROMPTS = INPUT_ROOT / "resources/protocol" / "prompts"
U.assert_no_sealed_inputs(
    [INPUT_ROOT] + list(OBS_DIRS) + list(EDIT_DIRS) + list(PRIOR_OUTPUT_DIRS)
)

In [ ]:
from medvqa import neutral_controls as controls

if ANSWERER != "Q" or tuple(CELLS) != controls.ARMS:
    raise ValueError("Neutral controls require Qwen and the three defined conditions.")
control_bank = controls.load_bank(source_root)
LABEL_MAP_FILE = Path(LABEL_MAP_FILE or INPUT_ROOT / "artifacts/logprob_label_map.json")
CORR_PLAN_FILE = Path(CORR_PLAN_FILE or INPUT_ROOT / "artifacts/corruption_plan.json")

In [ ]:
COHORTS = RUN.cohorts_for(MODE)
cells = controls.cell_specs()
needs_corr = any((c["intervention"] == "INT-CORR" for c in cells))
lock_files = RUN.prompt_lock_files(INPUT_ROOT, ["templates/answerer_template.txt"])
lock_files.update(
    RUN.prep_lock_files(
        INPUT_ROOT,
        [
            "route_metadata.json",
            "route_pairs.json",
            "images_index.json",
            "shuf_donors.json",
        ]
        + [f"cohorts/{c}.jsonl" for c in COHORTS],
    )
)
if needs_corr:
    lock_files["artifacts/corruption_plan.json"] = Path(CORR_PLAN_FILE)
if MODE == "TEST":
    lock_files["run_params.json"] = RUN_PARAMS_FILE
    lock_files["artifacts/logprob_label_map.json"] = Path(LABEL_MAP_FILE)
lock_report = U.verify_locked_inputs(LOCK_FILE, lock_files, MODE)
if MODE == "TEST":
    rp = U.read_json(RUN_PARAMS_FILE)
    if RUN.rp_get(rp, "precision", ANSWERER) != PRECISION:
        raise U.LockError("PRECISION differs from locked run_params.json")
    if ANSWERER == "M" and (
        RUN.rp_get(rp, "llavamed", "code_commit") != LLAVAMED_CODE_COMMIT
        or RUN.rp_get(rp, "llavamed", "conv_mode") != LLAVAMED_CONV_MODE
    ):
        raise U.LockError(
            "LLaVA-Med commit/conv_mode differ from locked run_params.json"
        )
print("cells:", [c["cell"] for c in cells], "| locked ok:", len(lock_report["files"]))

In [ ]:
rows = [
    row for row in RUN.load_cohort_rows(PREP, COHORTS) if row["answer_kind"] != "open"
]
nids = sorted({row["neutral_id"] for row in rows})
template = P.AnswerTemplate(PROMPTS / "templates/answerer_template.txt")
verified_units = controls.plan(rows, MODE, control_bank)
shuf_report = {"used": False}


def obs_text_for(condition, neutral_id):
    return controls.observation(control_bank, MODE, condition, neutral_id)


print(
    "Questions:", len(rows), "Images:", len(nids), "Answer units:", len(verified_units)
)
plan_sha = None

In [ ]:
if BACKEND == "stub":
    model = B.StubVLM(ANSWERER, PRECISION)
elif ANSWERER == "M":
    model = B.LlavaMedBackend(PRECISION, LLAVAMED_CONV_MODE)
else:
    model = B.HFVisionBackend(ANSWERER, PRECISION)
label_map = B.build_label_map(model.decode_one, model.vocab_size())
TOK_KEY = f"ANS-{ANSWERER}"
if MODE == "TEST":
    locked_map = U.read_json(LABEL_MAP_FILE)[TOK_KEY]["labels"]
    if locked_map != label_map:
        raise U.LockError("T(c) differs from locked logprob_label_map.json")
session = U.ShardSession(
    "neutral_controls",
    RUN_ID,
    MODE,
    OUT_ROOT,
    SHARD_INDEX,
    NUM_SHARDS,
    PRIOR_OUTPUT_DIRS,
    time_limit_s=SHARD_TIME_LIMIT_S,
    max_items=MAX_ITEMS_THIS_SESSION,
)
ART = OUT_ROOT / RUN_ID / "artifacts"
if MODE != "TEST":
    U.write_json(
        ART / f"logprob_label_map_{TOK_KEY}.json",
        {
            TOK_KEY: {
                "model_id": model.identity["model"],
                "revision": model.identity["revision"],
                "vocab_size": model.vocab_size(),
                "labels": label_map,
            }
        },
    )
run_cfg_sha = RUN.write_run_config(
    session,
    {
        "notebook": "neutral_controls",
        "platform": dict(PLATFORM_CFG, lock=platform_report),
        "design": DESIGN,
        "mode": MODE,
        "answerer": f"ANS-{ANSWERER}",
        "cells": [c["cell"] for c in cells],
        "model": model.identity,
        "precision": PRECISION,
        "answer_cap": CL.ANSWER_CAP,
        "generate_kwargs": B.GREEDY_KW,
        "generation_config_effective": model.identity.get(
            "generation_config_effective"
        ),
        "env_pins": env_report,
        "label_map_sha256": U.digest_json(label_map),
        "corruption_plan_sha256": plan_sha if needs_corr else None,
        "obs_text_insertion": "frozen neutral-control text without normalization",
        "shuf": shuf_report,
        "env": U.environment_info(),
        "locked_inputs": lock_report,
    },
)

In [ ]:
units = verified_units
keys = [u[0] for u in units]
shard_keys = set(U.plan_shard(keys, NUM_SHARDS, SHARD_INDEX))
done, _ = session.done_keys()
todo = [u for u in units if u[0] in shard_keys and u[0] not in done]
print(
    "answer turns planned:",
    len(keys),
    "| this shard:",
    len(shard_keys),
    "| to do:",
    len(todo),
)

In [ ]:
img_cache = {}
for key, c, r, o in todo:
    if session.should_stop():
        break
    nid, kind = (r["neutral_id"], r["answer_kind"])
    rec = session.base("ANS-v1", key)
    opts = (
        P.cyclic_order([x["text"] for x in r["options"]], o) if kind == "mcq" else None
    )
    rec.update(
        {
            "cell": c["cell"],
            "answerer": c["answerer"],
            "obs_observer": c["obs_observer"],
            "obs_arm": c["obs_arm"],
            "intervention": c["intervention"],
            "qid": r["qid"],
            "neutral_id": nid,
            "cohort": r["cohort"],
            "answer_kind": kind,
            "order": o if kind == "mcq" else None,
            "K": r["K"],
            "options_presented": opts,
            "image_condition": c["image_condition"],
            "max_new_tokens": CL.ANSWER_CAP[kind],
        }
    )
    text_obs, ref = obs_text_for(c, nid)
    rec["obs_ref"] = ref
    if text_obs is None:
        rec.update({"status": "skipped_int_missing", "text": None, "logprob": None})
        session.write(rec)
        continue
    if nid not in img_cache:
        img_cache = {nid: RUN.open_image(PREP, nid)}
    image = img_cache[nid]
    if image is None:
        rec.update({"status": "image_broken", "text": None, "logprob": None})
        session.write(rec)
        continue
    if c["image_condition"] == "gray128":
        image = I.gray_image(image)
    prompt = template.render(text_obs, r["question"], kind, opts)
    P.check6_answer_prompt(template, prompt, kind, len(opts or []))
    P.check3_no_original_names(prompt.replace(text_obs, ""))
    rec.update(
        {
            "obs_text_sha256": U.sha256_text(text_obs),
            "prompt_sha256": U.sha256_text(prompt),
        }
    )
    t0 = time.time()
    try:
        g = model.generate(image, prompt, CL.ANSWER_CAP[kind])
    except Exception as exc:
        rec.update(
            {
                "status": "error",
                "error": f"{type(exc).__name__}: {str(exc)[:300]}",
                "text": None,
            }
        )
        session.write(rec)
        continue
    lp = None
    if kind in ("yesno", "mcq"):
        labels = list(B.YESNO_LABELS) if kind == "yesno" else [x["label"] for x in opts]
        lp = B.label_logprobs(g["first_logits"], label_map, labels)
        lp["method"] = g["logits_method"]
    rec.update(
        {
            "status": "ok",
            "text": g["text"],
            "n_output_tokens": g["n_output_tokens"],
            "truncated": g["truncated"],
            "finish_reason": g["finish_reason"],
            "logprob": lp,
            "quality": U.quality_flags(g["text"]),
            "timing_s": round(time.time() - t0, 3),
        }
    )
    session.write(rec)

In [ ]:
allrows = list(
    RUN.load_records([OUT_ROOT / RUN_ID] + list(PRIOR_OUTPUT_DIRS), "ANS-v1").values()
)
quality = U.summarize_quality(
    [r for r in allrows if r["status"] == "ok"], ("cell", "answer_kind")
)
rep = session.integrity(
    sorted(shard_keys),
    {
        "checks": {
            "check3": "pass",
            "check4": "pass",
            "check5": "frozen control text",
            "check6": "pass",
            "check7": lock_report["verified"],
        },
        "quality": quality,
        "status_counts": dict(Counter((r["status"] for r in allrows))),
        "run_config_sha256": run_cfg_sha,
        "locked_inputs": {
            "verified": lock_report["verified"],
            "unlisted_used": lock_report["unlisted_used"],
        },
    },
)
print(
    {
        k: rep[k]
        for k in (
            "stopped_reason",
            "done_this_session",
            "done_total_in_shard",
            "remaining_in_shard",
            "errors",
            "duplicate_keys",
        )
    }
)
print("status:", rep["status_counts"])